In [4]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
X,y = make_regression(n_samples=2000, n_features=10, noise=10, random_state=42)


xtrain, xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
xtrain =  scaler.fit_transform(xtrain)
xtest = scaler.transform(xtest)

import torch
xtrain = torch.tensor (xtrain, dtype=torch.float32)
xtest = torch.tensor(xtest, dtype=torch.float32)
ytrain = torch.tensor(ytrain, dtype=torch.float32).reshape(-1, 1) * 0.
ytest = torch.tensor(ytest, dtype=torch.float32).reshape(-1, 1)

In [5]:
import torch.nn as nn
activators = {
"Sigmoid": nn.Sigmoid(),
"ReLU": nn. ReLU(),
"Tanh": nn. Tanh(),
"LeakyReLU": nn. LeakyReLU(),
"SiLU": nn.SiLU(),
"Swish": nn.SiLU(),
}

In [6]:
class Mlp(nn.Module):
    def __init__(
        self,
        input_size=10,
        hidden_size=32,
        output_size=1,
        activation="RelU"
):
        super(Mlp, self).__init__()
        self.activation = activators.get(activation,None)
        if self.activation is None :
            raise ValueError(activation)
        self.layers = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            self.activation,
            nn.Linear(hidden_size, output_size),
            
        )
    def forward(self, x) :
        return self.layers(x)

In [7]:
def train_model(model, X, y, epochs=1000, lr=0.01) :
    criterion = nn.functional.mse_loss
    optimaizer = torch.optim.SGD(model.parameters(), lr=lr)
    losses = [*range(epochs)]
    for epoch in range(epochs):
        output = model(X)
        loss = criterion(output, y)
        optimaizer.zero_grad()
        loss.backward()
        optimaizer.step()
        losses[epoch] = loss.item()
    return losses

In [8]:
mlps = {k: Mlp(activation=k) for k in activators.keys()}
#mlps

In [9]:
losses = {}
predictions = {}
for k, mlp in mlps.items():
    losses[k] = train_model(mlp, xtrain, ytrain)
    with torch.no_grad():
        predictions[k] = mlp(xtest)
print(*zip(*(predictions[k][:5] for k in mlps.keys()), ytest[:5]), sep='\n')

(tensor([317.1496]), tensor([nan]), tensor([263.7213]), tensor([nan]), tensor([nan]), tensor([nan]), tensor([297.7354]))
(tensor([-71.6107]), tensor([nan]), tensor([-76.2094]), tensor([nan]), tensor([nan]), tensor([nan]), tensor([-65.3510]))
(tensor([-137.5801]), tensor([nan]), tensor([-135.9117]), tensor([nan]), tensor([nan]), tensor([nan]), tensor([-123.7060]))
(tensor([377.7079]), tensor([nan]), tensor([356.1127]), tensor([nan]), tensor([nan]), tensor([nan]), tensor([364.9809]))
(tensor([199.8655]), tensor([nan]), tensor([190.3010]), tensor([nan]), tensor([nan]), tensor([nan]), tensor([197.0017]))


In [17]:
import sklearn.metrics as sm
from IPython.display import Markdown as md
md("\n".join((
    "|A|M|R|",
    "|-|-|-|",
    *(f"|{k}|{sm.mean_absolute_error(ytest, v)}|{sm.r2_score(ytest, v)}|" for k,v in predictions.items())
)))

ValueError: Input contains NaN.